In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import astropy.coordinates as coord
import astropy.units as u
import emcee
import sys
import fitsio
from scipy.special import factorial
from astropy.table import Table
from numpy.polynomial.polynomial import Polynomial
from sklearn.metrics import mean_squared_error
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.pipeline import Pipeline

if './SelfCalGroupFinder/py/' not in sys.path:
    sys.path.append('./SelfCalGroupFinder/py/')
from pyutils import *
import plotting as pp
from dataloc import *
from bgs_helpers import *
import catalog_definitions as cat
from groupcatalog import *

%load_ext autoreload
%autoreload 2

In [ ]:
#table = create_merged_file(BGS_Y3_ANY_FULL_FILE, IAN_BGS_Y3_MERGED_FILE_LOA, "3")
table = Table.read(IAN_BGS_Y3_MERGED_FILE_LOA)

In [ ]:
# Use a previous run of the group catalog, which has nice cuts applied to it, to get a clean sample of galaxies to do PCA on.
bgs_y3_pzp_2_6_c2 = deserialize(cat.bgs_y3_pzp_2_6_c2)
df = bgs_y3_pzp_2_6_c2.all_data
df = df.loc[df['Z_ASSIGNED_FLAG'] == AssignedRedshiftFlag.DESI_SPEC.value] # Only DESI observed ones of course
df = df.loc[:, ['TARGETID']] # Drop everything else from group catalog
print(len(df))

In [ ]:
names = [name for name in table.colnames if len(table[name].shape) <= 1] 
dft = table[names].to_pandas()

df = df.merge(dft, on='TARGETID', how='inner')
print(len(df))

In [ ]:
# Derive some columns to include in PCA
df['G-R'] = df['ABSMAG01_SDSS_G'] - df['ABSMAG01_SDSS_R']
df['G-Z'] = df['ABSMAG01_SDSS_G'] - df['ABSMAG01_SDSS_Z']
df['R-Z'] = df['ABSMAG01_SDSS_R'] - df['ABSMAG01_SDSS_Z']
df['SSFR'] = df['SFR'] / np.power(10, df['LOGMSTAR'])
#df['LOGSSFR'] = np.log10(df['SSFR'])
df['LOGSSFR'] = soft_clip_floor(np.log10(df['SSFR']), floor=-14)

# Set inf to na
df.replace([np.inf, -np.inf], np.nan, inplace=True)

## Examine properties

In [ ]:
print(df.columns)

In [ ]:
# Examine age in z bins
zbins = [0, 0.05, 0.1, 0.2, 0.3, 0.4, 0.5]
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for i in range(len(zbins)-1):
    zmin, zmax = zbins[i], zbins[i+1]
    print(f"Redshift bin: {zmin} - {zmax}")
    subset = df.loc[(df['Z'] >= zmin) & (df['Z'] < zmax)]
    # QUIESCENT vs STAR-FORMING
    r = subset.loc[subset['QUIESCENT'] == 1]
    b = subset.loc[subset['QUIESCENT'] == 0]
    axes[0].hist(r['AGE'], bins=50, label=f'{zmin} - {zmax}', histtype='step', lw=1.5)
    axes[1].hist(b['AGE'], bins=50, label=f'{zmin} - {zmax}', histtype='step', lw=1.5)
    
axes[0].set_title('Quiescent')
axes[1].set_title('Star-forming')
axes[0].set_xlabel('AGE')
axes[0].set_yscale('log')
axes[1].set_xlabel('AGE')
axes[1].set_yscale('log')
axes[1].legend()

In [ ]:
# Examine VISP in z bins
zbins = [0, 0.05, 0.1, 0.2, 0.3, 0.4, 0.5]
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for i in range(len(zbins)-1):
    zmin, zmax = zbins[i], zbins[i+1]
    print(f"Redshift bin: {zmin} - {zmax}")
    subset = df.loc[(df['Z'] >= zmin) & (df['Z'] < zmax)]
    # QUIESCENT vs STAR-FORMING
    r = subset.loc[subset['QUIESCENT'] == 1]
    b = subset.loc[subset['QUIESCENT'] == 0]
    axes[0].hist(r['VDISP'], bins=50, label=f'{zmin} - {zmax}', histtype='step', lw=1.5)
    axes[1].hist(b['VDISP'], bins=50, label=f'{zmin} - {zmax}', histtype='step', lw=1.5)


    # 250 is a special value in the FSF code. What percent of them are at 250?
    print(f"Percent of quiescent galaxies with VDISP=250 in this redshift bin: {(r['VDISP'] == 250).mean():.2%}")
    print(f"Percent of star-forming galaxies with VDISP=250 in this redshift bin: {(b['VDISP'] == 250).mean():.2%}")
    
axes[0].set_title('Quiescent')
axes[1].set_title('Star-forming')
axes[0].set_xlabel('VDISP')
axes[0].set_yscale('log')
axes[1].set_xlabel('VDISP')
axes[1].set_yscale('log')
axes[1].legend()


In [ ]:
# Examine VDISP in magnitude bins
magbins = np.arange(-24, -16, 1.0)
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for i in range(len(magbins)-1):
    magmin, magmax = magbins[i], magbins[i+1]
    print(f"Mag bin: {magmin} - {magmax}")
    subset = df.loc[(df['ABSMAG01_SDSS_R'] >= magmin) & (df['ABSMAG01_SDSS_R'] < magmax)]
    # QUIESCENT vs STAR-FORMING
    r = subset.loc[subset['QUIESCENT'] == 1]
    b = subset.loc[subset['QUIESCENT'] == 0]
    axes[0].hist(r['VDISP'], bins=50, label=f'{magmin} to {magmax}', histtype='step', lw=1.5)
    axes[1].hist(b['VDISP'], bins=50, label=f'{magmin} to {magmax}', histtype='step', lw=1.5)


    # 250 is a special value in the FSF code. What percent of them are at 250?
    print(f"Q : VDISP=250: {(r['VDISP'] == 250).mean():.2%}")
    print(f"SF: VDISP=250: {(b['VDISP'] == 250).mean():.2%}")
    
axes[0].set_title('Quiescent')
axes[1].set_title('Star-forming')
axes[0].set_xlabel('VDISP')
axes[0].set_yscale('log')
axes[1].set_xlabel('VDISP')
axes[1].set_yscale('log')
axes[1].legend()


In [ ]:
# Plot LOGSSFR for QUIESCENT vs STAR-FORMING to see if it looks reasonable
bins = np.linspace(-14, -8, 100)
plt.hist(df['LOGSSFR'], bins=bins, alpha=0.5, label='All')
plt.legend()
plt.yscale('log')
plt.title('LOGSSFR for Quiescent vs Star-forming')


In [ ]:
# Plot LOGSSFR for QUIESCENT vs STAR-FORMING to see if it looks reasonable
red = df.loc[df['QUIESCENT']]
blue = df.loc[~df['QUIESCENT']] 
bins = np.linspace(-14, -8, 100)
plt.hist(red['LOGSSFR'], bins=bins, alpha=0.5, label='Quiescent')
plt.hist(blue['LOGSSFR'], bins=bins, alpha=0.5, label='Star-forming')
plt.legend()
plt.yscale('log')
plt.title('LOGSSFR for Quiescent vs Star-forming')

In [ ]:
# Compute and show the correlation matrix of the features
feature_cols = ['ABSMAG01_SDSS_R', 'ABSMAG01_SDSS_G', 'ABSMAG01_SDSS_Z', 
                'LOGMSTAR', 'DN4000_MODEL', 'SHAPE_R_KPC', 'SERSIC', 'LOGSSFR',
                'AGE']
features = df[feature_cols].dropna()
feature_cols[0] = '$M_r$'
feature_cols[1] = '$M_g$'
feature_cols[2] = '$M_z$'
feature_cols[3] = '$\log M_*$'
feature_cols[4] = '$Dn4000$'
feature_cols[5] = '$R_e$ (kpc)'
feature_cols[6] = '$n_s$'
feature_cols[7] = '$\log SSFR$'
feature_cols[8] = '$LW Age$'
corr_matrix = np.corrcoef(features.T)  # or features.corr() as a DataFrame
plt.figure(figsize=(6, 5))
plt.imshow(corr_matrix, cmap='RdBu_r', vmin=-1, vmax=1, interpolation='none')
plt.colorbar()
plt.xticks(ticks=np.arange(len(feature_cols)), labels=feature_cols, rotation=45, ha='right')
plt.yticks(ticks=np.arange(len(feature_cols)), labels=feature_cols)

for i in range(len(feature_cols)):
    for j in range(len(feature_cols)):
        plt.text(j, i, f'{corr_matrix[i,j]:.2f}', ha='center', va='center', fontsize=9)
plt.title('Galaxy Correlation Matrix')
plt.tight_layout()
plt.show()

## PCA

In [ ]:
# Seperate into test/training sets
from sklearn.model_selection import train_test_split
train, test = train_test_split(df, test_size=0.1, random_state=894)

# TODO galaxy density in X mpc around each target (Andres)
# TODO Concentration (c90c50 column) (code ready on NERSC, put into merged file)
# VDISP is 250 km/s, the default value far too often to be used as a feature. 
# ZZSUN (metallicity) is pure junk unfortunately.

# TODO AGE is Light Weighted Age. It has two weird peaks at 6 and 12 Gyr. Shares info with Dn4000 a lot I think. Maybe it's still useful?

# TODO HALPHA_EW, HBETA_EW could try
# TODO Consider how to incorporate redshift 'Z'. Could do PCA on residuals after redshift trend removal?
feature_cols = ['ABSMAG01_SDSS_R', 'ABSMAG01_SDSS_G', 'ABSMAG01_SDSS_Z', 
                'LOGMSTAR', 'DN4000_MODEL', 'SHAPE_R_KPC', 'SERSIC', 'LOGSSFR',
                'AGE']
                #'G-R', 'G-Z', 'R-Z'] # This is redundant. Fine to include, but complicates interpretation of loadings. Maybe add back in later after understanding the main features.
n_features = len(feature_cols)

# Drop rows with any NaN in the selected features
train_clean = train.dropna(subset=feature_cols)
test_clean = test.dropna(subset=feature_cols)
print(f"Training samples after NaN drop: {len(train_clean)} / {len(train)}")
print(f"Test samples after NaN drop: {len(test_clean)} / {len(test)}")

In [ ]:
# Scale first - critical since features have very different units/ranges
scaler = StandardScaler()
train_scaled = scaler.fit_transform(train_clean[feature_cols])
test_scaled = scaler.transform(test_clean[feature_cols])

In [ ]:
# Compare the original components to the scaled ones
for f in feature_cols:
    plt.hist(train_clean[f], bins=100, alpha=0.5, label=f'Original')
    plt.hist(train_scaled[:, feature_cols.index(f)], bins=100, alpha=0.5, label=f'Scaled')
    plt.legend()
    plt.title(f'{f}')
    plt.yscale('log')
    plt.show()

In [ ]:
# Fit full PCA on training set
pca_full = PCA(n_components=n_features)
pca_full.fit(train_scaled)

# Explained variance
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(range(1, n_features+1), pca_full.explained_variance_ratio_, color='steelblue')
axes[0].set_xlabel('PCA Component')
axes[0].set_ylabel('Explained Variance Ratio')
axes[0].set_title('Variance per Component')

axes[1].plot(range(1, n_features+1), np.cumsum(pca_full.explained_variance_ratio_), 'ko-')
axes[1].axhline(0.95, color='red', linestyle='--', label='95%')
axes[1].axhline(0.99, color='orange', linestyle='--', label='99%')
axes[1].set_xlabel('Number of PCA Components')
axes[1].set_ylabel('Cumulative Explained Variance')
axes[1].set_title('Cumulative Explained Variance')
axes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
# Print feature loadings for top components
print("\nFeature loadings for top 6 components:")
loadings = pd.DataFrame(
    pca_full.components_[:6].T,
    index=feature_cols,
    columns=[f'PC{i+1}' for i in range(6)]
)
print(loadings.round(3).to_string())

In [ ]:
# Apply PCA transformation to training and test sets
train_pca_full = pca_full.transform(train_scaled)
test_pca_full = pca_full.transform(test_scaled)

# Join train with train_pca_full
train_with_pca = pd.concat([train_clean.reset_index(drop=True), pd.DataFrame(train_pca_full, columns=[f'PCA{i+1}' for i in range(train_pca_full.shape[1])])], axis=1)
test_with_pca = pd.concat([test_clean.reset_index(drop=True), pd.DataFrame(test_pca_full, columns=[f'PCA{i+1}' for i in range(test_pca_full.shape[1])])], axis=1)

In [ ]:
red = train_with_pca.loc[train_with_pca['QUIESCENT'] == 1]
blue = train_with_pca.loc[train_with_pca['QUIESCENT'] == 0]
bins = np.linspace(-5, 5, 100)
plt.hist(train_with_pca['PCA2'], bins=bins, alpha=1.0, label='All', color='k', histtype='step', linewidth=1.5)
plt.hist(red['PCA2'], bins=bins, alpha=0.5, label='Quiescent', color='red', histtype='step', linewidth=1.5)
plt.hist(blue['PCA2'], bins=bins, alpha=0.5, label='Star-forming', color='blue', histtype='step', linewidth=1.5)
plt.legend()
plt.title('PCA2 Distribution')

# Now do this in fixed magnitude bins to see if the separation is just due to magnitude or if it holds at fixed magnitude.
magnitude_bins = np.linspace(-24, -17, 8)  # Example magnitude bins
fig, axes = plt.subplots(1, 1, figsize=(8, 6))
for i in range(len(magnitude_bins)-1):
    magmin, magmax = magnitude_bins[i], magnitude_bins[i+1]
    print(f"Mag bin: {magmin} - {magmax}")
    subset = train_with_pca.loc[(train_with_pca['ABSMAG01_SDSS_R'] >= magmin) & (train_with_pca['ABSMAG01_SDSS_R'] < magmax)]
    plt.hist(subset['PCA2'], bins=bins, alpha=0.6, label=f'{magmin} to {magmax}', histtype='step', linewidth=1.5, density=True)
plt.legend()
plt.title('PCA2 Distribution in Magnitude Bins')

In [ ]:
# Show histograms of the first few PCA components to see if they look reasonable
for i in range(min(5, n_features)):
    plt.hist(train_pca_full[:, i], bins=100, alpha=0.5, label=f'Train PCA {i+1}', density=True)
    plt.hist(test_pca_full[:, i], bins=100, alpha=0.5, label=f'Test PCA {i+1}', density=True)
    plt.legend()
    plt.title(f'PCA Component {i+1} Distribution')
    plt.show()

## Autoencoder reconstruction as a function of components kept

In [ ]:
n_components_range = range(1, n_features + 1)
mse_train = []
mse_test = []

for n in n_components_range:
    pca_n = PCA(n_components=n)
    # Encode (project to n components) then decode (reconstruct)
    train_encoded = pca_n.fit_transform(train_scaled)
    train_reconstructed = pca_n.inverse_transform(train_encoded)
    test_encoded = pca_n.transform(test_scaled)
    test_reconstructed = pca_n.inverse_transform(test_encoded)

    mse_train.append(mean_squared_error(train_scaled, train_reconstructed))
    mse_test.append(mean_squared_error(test_scaled, test_reconstructed))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(n_components_range, mse_train, 'ko-', label='Train')
ax.plot(n_components_range, mse_test, 'rs-', label='Test')
ax.set_xlabel('Number of PCA Components Kept')
ax.set_ylabel('Reconstruction MSE (standardized units)')
ax.set_title('PCA Autoencoder Performance')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Print summary table
print(f"\n{'N Components':<15} {'Cumul. Var':<15} {'Train MSE':<15} {'Test MSE':<15}")
print("-" * 60)
cumvar = np.cumsum(pca_full.explained_variance_ratio_)
for i, n in enumerate(n_components_range):
    print(f"{n:<15} {cumvar[i]:<15.4f} {mse_train[i]:<15.4f} {mse_test[i]:<15.4f}")